# Ollama PDF RAG Notebook

## Install Required Packages
First, let's install all necessary packages for the RAG system.

In [13]:
!pip install -q -U langchain-core langchain-community langchain-ollama langchain-text-splitters chromadb pypdf
print("✓ All required packages installed successfully")

✓ All required packages installed successfully


DEPRECATION: Loading egg at c:\python313\lib\site-packages\vboxapi-1.0-py3.13.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


## Check Prerequisites
Make sure Ollama is running with required models installed.

In [14]:
# Check if Ollama is running
import subprocess
import sys

def check_ollama():
    try:
        result = subprocess.run(['ollama', 'list'], 
                              capture_output=True, 
                              text=True, 
                              timeout=5)
        if result.returncode == 0:
            print("✓ Ollama is running!")
            print("\nInstalled models:")
            print(result.stdout)
            return True
        else:
            print("❌ Ollama is not responding properly")
            return False
    except FileNotFoundError:
        print("❌ Ollama is not installed or not in PATH")
        print("   Please install Ollama from: https://ollama.ai")
        return False
    except subprocess.TimeoutExpired:
        print("❌ Ollama command timed out")
        return False
    except Exception as e:
        print(f"❌ Error checking Ollama: {str(e)}")
        return False

ollama_ready = check_ollama()
if not ollama_ready:
    print("\n⚠️  Please start Ollama before continuing!")
    print("   Required models: mistral, nomic-embed-text")
    print("   Install with: ollama pull mistral && ollama pull nomic-embed-text")

✓ Ollama is running!

Installed models:
NAME                        ID              SIZE      MODIFIED    
mxbai-embed-large:latest    468836162de7    669 MB    2 hours ago    
qwen2:7b                    dd314f039b9d    4.4 GB    2 hours ago    
mistral:latest              6577803aa9a0    4.4 GB    4 days ago     
nomic-embed-text:latest     0a109f422b47    274 MB    4 days ago     



## Import Libraries

In [15]:
# Imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama.chat_models import ChatOllama
from langchain_core.runnables import RunnablePassthrough

import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, Markdown
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


## Load PDF

In [16]:
# Load PDF using absolute path to avoid cwd issues
import os

local_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "Data", "PDF", "scammer-agent.pdf")
local_path = os.path.normpath(local_path)

# Fallback: try absolute path directly if relative doesn't resolve
if not os.path.exists(local_path):
    local_path = r"D:\3-2\Ollama\rag_project\Data\PDF\scammer-agent.pdf"

if os.path.exists(local_path):
    print(f"Loading PDF: {local_path}")
    loader = PyPDFLoader(file_path=local_path)
    data = loader.load()
    print(f"✓ PDF loaded successfully!")
    print(f"  Document has {len(data)} page(s)")
else:
    print(f"❌ PDF not found. Checked: {local_path}")
    print(f"   Current directory: {os.getcwd()}")

Loading PDF: d:\3-2\Ollama\rag_project\Data\PDF\scammer-agent.pdf
✓ PDF loaded successfully!
  Document has 5 page(s)


## Split text into chunks

In [17]:
# Split text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(data)
print(f"✓ Text split into {len(chunks)} chunks")
print(f"  Chunk size: 1000 characters")
print(f"  Chunk overlap: 200 characters")

✓ Text split into 23 chunks
  Chunk size: 1000 characters
  Chunk overlap: 200 characters


## Create vector database

In [18]:
# Create vector database
print("Creating vector database... (this may take a minute)")
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=OllamaEmbeddings(model="nomic-embed-text"),
    collection_name="local-rag"
)
print("✓ Vector database created successfully")
print(f"  Embedding model: nomic-embed-text")
print(f"  Collection: local-rag")

Creating vector database... (this may take a minute)
✓ Vector database created successfully
  Embedding model: nomic-embed-text
  Collection: local-rag


## Set up LLM and Retrieval

In [19]:
# Set up LLM and retrieval
local_model = "mistral" 
llm = ChatOllama(model=local_model)
print(f"✓ LLM initialized")
print(f"  Model: {local_model}")
print(f"  Make sure Ollama is running with this model installed!")

✓ LLM initialized
  Model: mistral
  Make sure Ollama is running with this model installed!


In [20]:
# Set up retriever (using similarity search with top-k results)
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)
print("✓ Retriever configured (top-5 similarity search)")

✓ Retriever configured (top-5 similarity search)


## Create chain

In [21]:
# RAG prompt template
template = """Answer the question based ONLY on the following context:
{context}
Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
print("✓ RAG prompt template created")

✓ RAG prompt template created


In [22]:
# Create chain
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print("✓ RAG chain created successfully")
print("  Ready to chat with your PDF!")

✓ RAG chain created successfully
  Ready to chat with your PDF!


## Chat with PDF

In [23]:
def chat_with_pdf(question):
    """
    Chat with the PDF using the RAG chain.
    """
    try:
        print(f"\n🤔 Question: {question}\n")
        print("💭 Thinking...")
        response = chain.invoke(question)
        print("\n📝 Answer:")
        print("-" * 50)
        display(Markdown(response))
        print("-" * 50)
        return response
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        print("   Make sure Ollama is running and the required models are installed.")
        return None

In [24]:
# Example 1
chat_with_pdf("What is the main idea of this document?")


🤔 Question: What is the main idea of this document?

💭 Thinking...

📝 Answer:
--------------------------------------------------


 The main idea of this document is the exploration of the capabilities of AI technology in performing common scams autonomously. The document discusses the design of agents that can navigate websites, retrieve user credentials, and perform actions necessary for scams, such as transferring money from bank accounts. It also mentions previous studies on dual use of AI technology and examples of real-world applications of AI in scams and spam. However, the authors do not release their code to prevent nefarious actors from leveraging their work.

--------------------------------------------------


' The main idea of this document is the exploration of the capabilities of AI technology in performing common scams autonomously. The document discusses the design of agents that can navigate websites, retrieve user credentials, and perform actions necessary for scams, such as transferring money from bank accounts. It also mentions previous studies on dual use of AI technology and examples of real-world applications of AI in scams and spam. However, the authors do not release their code to prevent nefarious actors from leveraging their work.'

In [25]:
# Example 2
chat_with_pdf("What is the purpose of the scammer agent?")


🤔 Question: What is the purpose of the scammer agent?

💭 Thinking...

📝 Answer:
--------------------------------------------------


 The purpose of the Scammer agent, as described in the provided context, is to autonomously perform common scams such as stealing bank credentials and transferring money by accessing bank websites, completing two-factor authentication processes, and other necessary actions. It uses a voice-enabled AI (GPT-4o) and a set of tools for performing these tasks. The developers designed this agent to highlight the capabilities of new technology, but they did not make it public to prevent nefarious actors from leveraging their work.

--------------------------------------------------


' The purpose of the Scammer agent, as described in the provided context, is to autonomously perform common scams such as stealing bank credentials and transferring money by accessing bank websites, completing two-factor authentication processes, and other necessary actions. It uses a voice-enabled AI (GPT-4o) and a set of tools for performing these tasks. The developers designed this agent to highlight the capabilities of new technology, but they did not make it public to prevent nefarious actors from leveraging their work.'

In [26]:
# Example 3
chat_with_pdf("Can you explain the case study highlighted in the document?")


🤔 Question: Can you explain the case study highlighted in the document?

💭 Thinking...

📝 Answer:
--------------------------------------------------


 The document describes a case study on AI agents designed to perform actions necessary for common scams. These agents consist of a base, voice-enabled Language Model (LLM), a set of tools that the LLM can use, and scam-specific instructions.

The agents have access to five browser access tools:
1. get_html: retrieves the HTML of a page.
2. navigate: navigates to a specific URL.
3. click_element: clicks on an element with a CSS selector.
4. fill_element: fills an element with the specified value.
5. evaluate_javascript: executes JavaScript on a page.

The agents were used in various scams, and one example provided is a phishing scam targeting Bank of America users. In this scenario, the agent navigates to the Bank of America login page, inputs the username and password (taking 6 actions), and then requests the 2FA code from the victim. After receiving the 2FA code, the agent performs additional actions to fill out the code, navigate to the transfer page, select a recipient, and ultimately transfer money.

The creators of these agents believe that their work highlights important capabilities of new technology, although they acknowledge the potential for malicious actors to take advantage of such technology. They also emphasize that they have chosen not to make their agents public due to concerns about their dual-use capabilities and the potential for nefarious actors to leverage their work.

--------------------------------------------------


' The document describes a case study on AI agents designed to perform actions necessary for common scams. These agents consist of a base, voice-enabled Language Model (LLM), a set of tools that the LLM can use, and scam-specific instructions.\n\nThe agents have access to five browser access tools:\n1. get_html: retrieves the HTML of a page.\n2. navigate: navigates to a specific URL.\n3. click_element: clicks on an element with a CSS selector.\n4. fill_element: fills an element with the specified value.\n5. evaluate_javascript: executes JavaScript on a page.\n\nThe agents were used in various scams, and one example provided is a phishing scam targeting Bank of America users. In this scenario, the agent navigates to the Bank of America login page, inputs the username and password (taking 6 actions), and then requests the 2FA code from the victim. After receiving the 2FA code, the agent performs additional actions to fill out the code, navigate to the transfer page, select a recipient, a

In [27]:
# Ask your own question here
your_question = "What are the key findings in this document?"
chat_with_pdf(your_question)


🤔 Question: What are the key findings in this document?

💭 Thinking...

📝 Answer:
--------------------------------------------------


 The key findings in this document include:

1. The paper discusses the ethical and social risks posed by language models, focusing on their potential misuse in scams.

2. It mentions several ways that AI agents can be used to perform scams, such as filling out specific fields, clicking on buttons, navigating to specific websites, and exploiting zero-day vulnerabilities.

3. Improvements in language models have led to broad improvements in various downstream tasks, and it is anticipated that this will also be the case for efficacy in scams.

4. The document outlines several studies related to the dual use of AI technology, including the potential for AI agents to autonomously hack websites and exploit programmatic behavior of language models for malicious purposes.

5. The paper emphasizes the importance of studying the capabilities of new technology, particularly in its dual use capabilities, but also notes that it's beneficial not to release the code so that nefarious actors cannot leverage the work.

6. The document does not make the agents public for two reasons: to prevent nefarious actors from using the work and to allow model providers (e.g., OpenAI) to build defenses against such nefarious use.

--------------------------------------------------


" The key findings in this document include:\n\n1. The paper discusses the ethical and social risks posed by language models, focusing on their potential misuse in scams.\n\n2. It mentions several ways that AI agents can be used to perform scams, such as filling out specific fields, clicking on buttons, navigating to specific websites, and exploiting zero-day vulnerabilities.\n\n3. Improvements in language models have led to broad improvements in various downstream tasks, and it is anticipated that this will also be the case for efficacy in scams.\n\n4. The document outlines several studies related to the dual use of AI technology, including the potential for AI agents to autonomously hack websites and exploit programmatic behavior of language models for malicious purposes.\n\n5. The paper emphasizes the importance of studying the capabilities of new technology, particularly in its dual use capabilities, but also notes that it's beneficial not to release the code so that nefarious acto

In [ ]:
# Optional: Clean up when done 
# ONLY RUN THIS when you're completely finished!
try:
    vector_db.delete_collection()
    print("✓ Vector database deleted successfully")
except Exception as e:
    print(f"Error deleting vector database: {str(e)}")

Vector database deleted successfully
